## Exercise 3 (1 point)
Write an MCP server running in streaming HTTP mode, exposing two tools:

get_current_date - returns current date in the format "Year-Month-Day" (YYYY-MM-DD)
get_current_datetime - returns current date and time in ISO 8601 format (up to seconds), i.e., YYYY-MM-DDTHH:MM:SS
Run it on port 8002.

# ALL THREE MCP SERVERS ARE IN SEPARATE .PY FILES.

## Note: I updated the system prompt in MCPManager to force tools usage.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.environ.get("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash-lite"  # 20 req/day free tier


In [2]:
import asyncio
import json
from contextlib import AsyncExitStack
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


class MCPManager:
    def __init__(self, servers: dict[str, str]):
        self.servers = servers
        self.clients = {}
        self.tools = []  # in OpenAI format
        self._stack = AsyncExitStack()

    async def __aenter__(self):
        for url in self.servers.values():
            read, write, session_id = await self._stack.enter_async_context(
                streamable_http_client(url)
            )
            session = await self._stack.enter_async_context(ClientSession(read, write))
            await session.initialize()

            tools_resp = await session.list_tools()
            for t in tools_resp.tools:
                self.clients[t.name] = session
                self.tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema,
                        },
                    }
                )

        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._stack.aclose()

    async def call_tool(self, name: str, args: dict) -> dict | str:
        result = await self.clients[name].call_tool(name, arguments=args)
        return result.content[0].text


async def make_llm_request_mcp(prompt: str) -> str:
    mcp_servers = {
        "weather_forecast": "http://localhost:8001/mcp",
        "date_time_server": "http://localhost:8002/mcp",
    }

    gemini_client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    async with MCPManager(mcp_servers) as mcp:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant connected to weather and date tools. Follow these strict rules:\n"
                    "1. ALWAYS assume the most prominent, famous global city when a user gives an ambiguous city name (e.g., Warsaw = Poland, Birmingham = UK). Do NOT ask the user for the country.\n"
                    "2. If a user asks for weather on a relative date (e.g., 'in two weeks', 'tomorrow'), you MUST autonomously call the date tool FIRST to find out today's date, calculate the target date yourself, and then call the weather tool. Do NOT ask the user what day it is."
                ),
            },
            {"role": "user", "content": prompt},
        ]

        for _ in range(10):
            response = gemini_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=mcp.tools,
                tool_choice="auto",
                max_completion_tokens=1000,
            )

            resp_message = response.choices[0].message
            if not resp_message.tool_calls:
                return resp_message.content

            messages.append(
                {k: v for k, v in resp_message.model_dump().items() if v is not None}
            )
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f"Executing tool '{func_name}'")
                func_result = await mcp.call_tool(func_name, func_args)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": str(func_result),
                    }
                )


In [3]:
prompt = "What will be weather in Birmingham in two weeks?"
response = await make_llm_request_mcp(prompt)
print("Response:\n", response)

print()

prompt = "What will be weather in Warsaw the day after tomorrow?"
response = await make_llm_request_mcp(prompt)
print("Response:\n", response)

print()

prompt = "What will be weather in New York in two months?"
response = await make_llm_request_mcp(prompt)
print("Response:\n", response)

Executing tool 'get_current_date'
Executing tool 'get_weather_forecast'
Response:
 The weather in Birmingham, UK in two weeks will be Fog and rain.

Executing tool 'get_current_date'
Executing tool 'get_weather_forecast'
Response:
 The weather in Warsaw will be sunshine the day after tomorrow.

Executing tool 'get_current_date'
Executing tool 'get_weather_forecast'
Response:
 The weather in New York in two months will be Sunshine.
